# 06 — Hyperparameter Tuning

**Joint notebook**, same CV harness and folds as notebook 05. Tune with `RandomizedSearchCV` (`scoring="average_precision"`, `random_state=RANDOM_STATE`), about 30-50 iterations.

| Owner | Model | Tuning focus |
| --- | --- | --- |
| Meegasthanna | Logistic Regression | `C`, `penalty`, `class_weight` |
| Bandara | XGBoost / LightGBM | `n_estimators`, `max_depth`, `learning_rate`, `scale_pos_weight` |
| Seneviratne | Random Forest | `n_estimators`, `max_depth`, `min_samples_leaf`, `class_weight` |
| Umer | SVM (RBF) | `C`, `gamma`, `class_weight` |

Umer also owns the SMOTE-vs-class-weights ablation and decision-threshold tuning (cross-cutting, applies to whichever model ends up selected).

**Output:** the results table from 05 extended with tuned rows (baseline vs tuned comparison).

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

In [2]:
import json

import numpy as np
import pandas as pd
from scipy.stats import randint, uniform

from sklearn.model_selection import RandomizedSearchCV, StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
import xgboost as xgb

from src.config import RANDOM_STATE, PROCESSED_DATA_DIR
from src.pipeline import build_preprocessing_pipeline
from src.evaluate import evaluate_cv, METRIC_NAMES

## Load the same train split and CV folds as notebook 05

TODO: reuse the exact same `StratifiedGroupKFold` configuration (same `random_state`, same `n_splits`) so baseline and tuned results are comparable.

In [3]:
train_df = pd.read_parquet(PROCESSED_DATA_DIR / "train.parquet")

with open(PROCESSED_DATA_DIR / "eligible_features.json") as f:
    eligible_features = json.load(f)

binary_cols = ["grip_lost"]
continuous_cols = [c for c in eligible_features if c not in binary_cols]

X_train = train_df[eligible_features]
y_train = train_df["Robot_ProtectiveStop"]
groups = train_df["cycle"]

# Same fold configuration as notebook 05, so baseline and tuned results are comparable
cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

X_train.shape, y_train.mean()

((5468, 16), 0.03108997805413314)

## Results table (baseline vs tuned)

TODO: load the baseline rows from notebook 05's output (re-run, or save/reload results as a small artifact in `data/processed/`), then append a tuned row per model here.

In [4]:
result_columns = ["model", "stage"] + [f"{m}_mean" for m in METRIC_NAMES] + [f"{m}_std" for m in METRIC_NAMES] + ["best_params"]
results = pd.DataFrame(columns=result_columns)

def add_result(model_name, stage, scores, best_params=None):
    row = {"model": model_name, "stage": stage, "best_params": best_params, **scores}
    results.loc[len(results)] = row

results

,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std,best_params


## Logistic Regression tuning — Meegasthanna

In [5]:
logreg_param_dist = {
    # TODO: "C": ..., "penalty": ..., "class_weight": ...
}
# TODO: RandomizedSearchCV(LogisticRegression(random_state=RANDOM_STATE), logreg_param_dist,
#   n_iter=40, scoring="average_precision", cv=cv, random_state=RANDOM_STATE)

## XGBoost / LightGBM tuning — Bandara

In [6]:
xgb_param_dist = {
    "classifier__n_estimators": randint(100, 500),
    "classifier__max_depth": randint(3, 10),
    "classifier__learning_rate": uniform(0.01, 0.29),
    "classifier__scale_pos_weight": uniform(1, 30),
}

xgb_search_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric="aucpr")),
])

xgb_search = RandomizedSearchCV(
    xgb_search_pipeline, xgb_param_dist, n_iter=40,
    scoring="average_precision", cv=cv, random_state=RANDOM_STATE, n_jobs=-1,
)
xgb_search.fit(X_train, y_train, groups=groups)

print("Best CV PR-AUC (search):", xgb_search.best_score_)
print("Best params:", xgb_search.best_params_)

# Re-run through evaluate_cv with the best params for the full metric set (mean+std),
# so this row is directly comparable to the baseline row's format.
best_xgb_params = {k.replace("classifier__", ""): v for k, v in xgb_search.best_params_.items()}
xgb_tuned_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", xgb.XGBClassifier(random_state=RANDOM_STATE, eval_metric="aucpr", **best_xgb_params)),
])
xgb_tuned_scores = evaluate_cv(xgb_tuned_pipeline, X_train, y_train, groups)
add_result("XGBoost", "tuned", xgb_tuned_scores, best_params=best_xgb_params)
results

Best CV PR-AUC (search): 0.4732048121862384
Best params: {'classifier__learning_rate': 0.04540550686319527, 'classifier__max_depth': 3, 'classifier__n_estimators': 330, 'classifier__scale_pos_weight': 28.204853246372622}


,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std,best_params
0,XGBoost,tuned,0.473205,0.689142,0.441291,0.512834,0.828957,0.923935,0.074298,0.159186,0.093766,0.040011,0.072297,0.048781,"{'learning_rate': 0.04540550686319527, 'max_de..."


## Random Forest tuning — Seneviratne

In [7]:
rf_param_dist = {
    "classifier__n_estimators": randint(100, 500),
    "classifier__max_depth": randint(3, 20),
    "classifier__min_samples_leaf": randint(1, 20),
    "classifier__class_weight": ["balanced", "balanced_subsample"],
}

rf_search_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE)),
])

rf_search = RandomizedSearchCV(
    rf_search_pipeline, rf_param_dist, n_iter=40,
    scoring="average_precision", cv=cv, random_state=RANDOM_STATE, n_jobs=-1,
)
rf_search.fit(X_train, y_train, groups=groups)

print("Best CV PR-AUC (search):", rf_search.best_score_)
print("Best params:", rf_search.best_params_)

best_rf_params = {k.replace("classifier__", ""): v for k, v in rf_search.best_params_.items()}
rf_tuned_pipeline = Pipeline([
    ("preprocessing", build_preprocessing_pipeline(continuous_cols, binary_cols)),
    ("classifier", RandomForestClassifier(random_state=RANDOM_STATE, **best_rf_params)),
])
rf_tuned_scores = evaluate_cv(rf_tuned_pipeline, X_train, y_train, groups)
add_result("Random Forest", "tuned", rf_tuned_scores, best_params=best_rf_params)
results

Best CV PR-AUC (search): 0.4386408601865659
Best params: {'classifier__class_weight': 'balanced_subsample', 'classifier__max_depth': 15, 'classifier__min_samples_leaf': 7, 'classifier__n_estimators': 286}


,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std,best_params
0,XGBoost,tuned,0.473205,0.689142,0.441291,0.512834,0.828957,0.923935,0.074298,0.159186,0.093766,0.040011,0.072297,0.048781,"{'learning_rate': 0.04540550686319527, 'max_de..."
1,Random Forest,tuned,0.438641,0.498633,0.412888,0.431612,0.737704,0.938454,0.110591,0.159130,0.087040,0.081180,0.075512,0.043814,"{'class_weight': 'balanced_subsample', 'max_de..."


## SVM (RBF) tuning — Umer

In [8]:
svm_param_dist = {
    # TODO: "C": ..., "gamma": ..., "class_weight": ...
}
# TODO: RandomizedSearchCV(SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE),
#   svm_param_dist, n_iter=40, scoring="average_precision", cv=cv, random_state=RANDOM_STATE)

## SMOTE vs class weights — Umer

TODO: for the leading model(s), compare `class_weight="balanced"` against SMOTE-resampling the training folds only (never the validation fold — resample inside the CV loop, not before it, or it leaks).

In [9]:
# TODO: SMOTE vs class_weight ablation

## Decision-threshold tuning — Umer

TODO: sweep the classification threshold on validation-fold predictions (not test) and pick the operating point that best trades recall vs false-alarm rate for this safety use case.

In [10]:
# TODO: threshold sweep + precision/recall curve

## Baseline vs tuned comparison

TODO: side-by-side table, one row pair (baseline, tuned) per model.

In [11]:
results.sort_values("pr_auc_mean", ascending=False)

,model,stage,pr_auc_mean,recall_mean,precision_mean,f1_mean,balanced_accuracy_mean,roc_auc_mean,pr_auc_std,recall_std,precision_std,f1_std,balanced_accuracy_std,roc_auc_std,best_params
0,XGBoost,tuned,0.473205,0.689142,0.441291,0.512834,0.828957,0.923935,0.074298,0.159186,0.093766,0.040011,0.072297,0.048781,"{'learning_rate': 0.04540550686319527, 'max_de..."
1,Random Forest,tuned,0.438641,0.498633,0.412888,0.431612,0.737704,0.938454,0.110591,0.159130,0.087040,0.081180,0.075512,0.043814,"{'class_weight': 'balanced_subsample', 'max_de..."


## Decision log

### Decision (Bandara): XGBoost tuning — PR-AUC 0.418→0.473, recall 0.307→0.689
- **Evidence:** `RandomizedSearchCV` (40 iterations, `scoring="average_precision"`, same `StratifiedGroupKFold` folds as the baseline) found `scale_pos_weight≈28.2` as the dominant lever — notably close to the true 25.5:1 majority-to-minority ratio confirmed in notebook 01 (7,077/278), not a value picked at random. This drove recall from 0.307 to 0.689, at the cost of precision (0.552→0.441).
- **Alternative considered:** Leaving `scale_pos_weight` at the notebook-05 baseline default (1) and tuning only `n_estimators`/`max_depth`/`learning_rate`.
- **Why rejected:** Recall is the metric that matters most for a safety system — missing a real protective stop is far costlier than an extra false alarm — so a tuning search that let `scale_pos_weight` move toward the data's actual imbalance ratio was the right choice, even though it costs some precision.

### Decision (Bandara): Random Forest tuning — PR-AUC 0.428→0.439, recall 0.161→0.499
- **Evidence:** Same search setup; found `class_weight="balanced_subsample"` (not plain `"balanced"`), `max_depth=15`, `min_samples_leaf=7`. Recall roughly tripled (0.161→0.499) versus the baseline, though it still trails XGBoost's tuned recall by a wide margin (0.499 vs 0.689).
- **Alternative considered:** Restricting the search to `class_weight="balanced"` only (matching the baseline setting) rather than including `"balanced_subsample"` in the search space.
- **Why rejected:** `balanced_subsample` recomputes class weights per bootstrap sample rather than once globally, which the search preferred — worth including both options rather than assuming the baseline's default was already best.

### Decision (Bandara): XGBoost over Random Forest, after tuning
- **Evidence:** Tuned XGBoost beats tuned Random Forest on both PR-AUC (0.473 vs 0.439) and recall (0.689 vs 0.499) — not a close call or a metric trade-off, XGBoost wins on the metric that matters most for this problem.
- **Alternative considered:** Waiting for all four models' tuned results (Logistic Regression, SVM still pending) before drawing any conclusion.
- **Why noted now, not rejected:** This isn't the final model decision — that's notebook 07, after every model is tuned and the whole team compares at the 29 Sep results meeting. But it's worth flagging early: XGBoost is the front-runner so far by a clear margin, not a marginal one.